# 12 — Language Models
**Goal:** Modelar la probabilidad del lenguaje — la base de todo NLP moderno.

## La idea
Un language model asigna una probabilidad a cualquier secuencia de palabras:

$$P(\text{la app está lenta}) = P(\text{la}) \cdot P(\text{app}|\text{la}) \cdot P(\text{está}|\text{la app}) \cdot P(\text{lenta}|\text{la app está})$$

**N-grama LM**: aproximar con una ventana de $n-1$ palabras anteriores:
$$P(w_t | w_1, ..., w_{t-1}) \approx P(w_t | w_{t-n+1}, ..., w_{t-1})$$

**Perplexity**: métrica de qué tan bien el modelo predice texto nuevo (menor = mejor):
$$PP = 2^{-\frac{1}{N}\sum_{i=1}^N \log_2 P(w_i | w_{i-1},...)}$$

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from itertools import chain

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return ['<s>'] + text.split() + ['</s>']  # tokens de inicio/fin

In [ ]:
# Corpus de entrenamiento
train_corpus = [
    'La app está muy lenta y tarda en cargar',
    'La aplicación no carga, da error al abrir',
    'La app es rápida y fácil de usar',
    'No me llegó el OTP al celular',
    'El código OTP no llega por SMS',
    'El OTP llegó rápido y sin problemas',
    'La carga de documentos falla constantemente',
    'No puedo subir los documentos requeridos',
    'Subir documentos fue muy sencillo',
    'Aprobaron mi crédito en minutos',
    'La evaluación de crédito lleva días',
    'El proceso de aprobación fue muy rápido',
    'El soporte no responde mis mensajes',
    'La atención al cliente fue excelente',
    'Nadie del soporte resuelve mis problemas',
    'Excelente servicio, recomiendo la tarjeta',
    'Increíbles beneficios, muy satisfecho',
    'Pésima experiencia, no recomiendo para nada',
] * 10  # repetir para más señal estadística

tokenized = [tokenize(s) for s in train_corpus]
all_tokens = list(chain.from_iterable(tokenized))
vocab = Counter(all_tokens)
print(f'Tokens totales: {len(all_tokens):,}')
print(f'Vocabulario:    {len(vocab)} palabras únicas')

## 1. Modelo de unigramas

In [ ]:
class UnigramLM:
    def __init__(self):
        self.counts = Counter()
        self.total  = 0

    def train(self, tokenized_corpus):
        for tokens in tokenized_corpus:
            self.counts.update(tokens)
        self.total = sum(self.counts.values())

    def prob(self, word):
        return self.counts.get(word, 0) / self.total

    def log_prob(self, word, eps=1e-10):
        return np.log2(self.prob(word) + eps)

    def sentence_prob(self, tokens):
        return np.prod([self.prob(t) for t in tokens if t not in ('<s>', '</s>')])

    def perplexity(self, tokenized_test):
        test_tokens = [t for tokens in tokenized_test
                       for t in tokens if t not in ('<s>', '</s>')]
        log_probs = [self.log_prob(t) for t in test_tokens]
        return 2 ** (-np.mean(log_probs))

lm1 = UnigramLM()
lm1.train(tokenized)

# Top palabras por probabilidad
print('Top 10 palabras (excluyendo marcadores):')
for word, count in vocab.most_common(12):
    if word not in ('<s>', '</s>'):
        print(f'  P("{word}") = {lm1.prob(word):.4f}')

## 2. Modelo de bigramas con Laplace smoothing

In [ ]:
class NgramLM:
    """N-gram language model with Laplace (add-k) smoothing."""

    def __init__(self, n=2, k=1.0):
        self.n = n
        self.k = k              # smoothing parameter
        self.ngram_counts   = Counter()
        self.context_counts = Counter()
        self.vocab          = set()

    def train(self, tokenized_corpus):
        for tokens in tokenized_corpus:
            self.vocab.update(tokens)
            for i in range(len(tokens) - self.n + 1):
                ngram   = tuple(tokens[i:i+self.n])
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
        self.V = len(self.vocab)

    def prob(self, word, context):
        """P(word | context) with Laplace smoothing."""
        context = tuple(context[-(self.n-1):])
        ngram   = context + (word,)
        num = self.ngram_counts[ngram] + self.k
        den = self.context_counts[context] + self.k * self.V
        return num / den

    def sentence_log_prob(self, tokens):
        lp = 0.0
        for i in range(self.n-1, len(tokens)):
            context = tokens[i-self.n+1:i]
            lp += np.log2(self.prob(tokens[i], context))
        return lp

    def perplexity(self, tokenized_test):
        total_log_prob = 0.0
        total_tokens   = 0
        for tokens in tokenized_test:
            lp = self.sentence_log_prob(tokens)
            n_tokens = len(tokens) - (self.n - 1)
            total_log_prob += lp
            total_tokens   += n_tokens
        return 2 ** (-total_log_prob / total_tokens)

    def next_word_probs(self, context, top_k=5):
        context = list(context)
        probs = {w: self.prob(w, context) for w in self.vocab
                 if w not in ('<s>',)}
        return sorted(probs.items(), key=lambda x: -x[1])[:top_k]

# Entrenar bigramas y trigramas
lm2 = NgramLM(n=2, k=1.0)
lm3 = NgramLM(n=3, k=0.1)
lm2.train(tokenized)
lm3.train(tokenized)

# Test
test_corpus = [tokenize(s) for s in [
    'La app está lenta',
    'No me llegó el OTP',
    'Excelente servicio muy rápido',
]]

print(f'Perplexity — Unigrama: {lm1.perplexity(test_corpus):.1f}')
print(f'Perplexity — Bigrama:  {lm2.perplexity(test_corpus):.1f}')
print(f'Perplexity — Trigrama: {lm3.perplexity(test_corpus):.1f}')
print('(menor perplexity = el modelo predice mejor el texto de test)')

## 3. Predicción de siguiente palabra

In [ ]:
queries = [
    ['la', 'app'],
    ['el', 'codigo'],
    ['no', 'me'],
    ['excelente'],
]

print('Predicción de siguiente palabra (bigrama):\n')
for context in queries:
    preds = lm2.next_word_probs(context, top_k=5)
    print(f'  "{" ".join(context)} ___"')
    for word, prob in preds:
        if word not in ('<s>', '</s>', 'la', 'el', 'de', 'y', 'a'):
            bar = '█' * int(prob * 100)
            print(f'    {word:20s} {prob:.3f} {bar}')
    print()

## 4. Generación de texto

In [ ]:
def generate(lm, seed_context, max_tokens=15, temperature=1.0, seed=42):
    """Genera texto muestreando del LM con temperatura."""
    rng = np.random.default_rng(seed)
    tokens = list(seed_context)
    for _ in range(max_tokens):
        preds = lm.next_word_probs(tokens, top_k=20)
        words  = [w for w, _ in preds]
        logits = np.array([p for _, p in preds])
        # Temperatura: >1 → más aleatorio, <1 → más determinista
        logits = logits ** (1 / temperature)
        probs  = logits / logits.sum()
        next_word = rng.choice(words, p=probs)
        if next_word == '</s>':
            break
        tokens.append(next_word)
    return ' '.join(tokens)

seeds = [['la', 'app'], ['el', 'soporte'], ['excelente']]
for seed in seeds:
    print(f'Seed: {seed}')
    print(f'  T=0.5: {generate(lm2, seed, temperature=0.5)}')
    print(f'  T=1.0: {generate(lm2, seed, temperature=1.0)}')
    print(f'  T=2.0: {generate(lm2, seed, temperature=2.0)}')
    print()

## 5. Detección de texto anómalo — perplexity como señal

In [ ]:
# Un LM entrenado en texto normal → perplexity alta en texto inusual
# Aplicación: detectar comentarios con idioma inusual / spam

test_sentences = [
    ('La app no carga correctamente',       'normal'),
    ('El OTP no llega al celular',          'normal'),
    ('Proceso de solicitud muy rápido',     'normal'),
    ('buy cheap medication online click',   'spam'),
    ('xyzqwerty blorp floop snarg',         'nonsense'),
    ('la tarjeta tarjeta tarjeta tarjeta',  'repetitivo'),
    ('Aprobación crediticia evaluación',    'normal'),
]

print(f'{"Texto":<50} {"Tipo":>10}  {"Perplexity":>12}')
print('-' * 77)
for text, tipo in test_sentences:
    pp = lm2.perplexity([tokenize(text)])
    flag = ' ⚠️' if pp > 100 else ''
    print(f'{text:<50} {tipo:>10}  {pp:>12.1f}{flag}')

## 6. Smoothing — el problema del cero

In [ ]:
# Sin smoothing, P(w | context_no_visto) = 0 → log prob = -∞
lm_no_smooth = NgramLM(n=2, k=0)  # sin Laplace
lm_no_smooth.train(tokenized)

lm_k01  = NgramLM(n=2, k=0.01)
lm_k1   = NgramLM(n=2, k=1.0)
lm_k10  = NgramLM(n=2, k=10.0)

for lm in [lm_k01, lm_k1, lm_k10]:
    lm.train(tokenized)

# Palabra no vista en el contexto "[soporte] rápido"
context_new = ['soporte']
word_new    = 'rápido'     # ¿aparece después de 'soporte' en el corpus?

print(f'P("{word_new}" | "{context_new[0]}")')
print(f'  sin smoothing (k=0):  {lm_no_smooth.prob(word_new, context_new):.6f}')
print(f'  Laplace k=0.01:       {lm_k01.prob(word_new, context_new):.6f}')
print(f'  Laplace k=1:          {lm_k1.prob(word_new, context_new):.6f}')
print(f'  Laplace k=10:         {lm_k10.prob(word_new, context_new):.6f}')
print()

# Comparar perplexity en test
print('Perplexity en test:')
for name, lm in [('k=0.01', lm_k01), ('k=1', lm_k1), ('k=10', lm_k10)]:
    print(f'  {name}: {lm.perplexity(test_corpus):.1f}')

## Resumen
| Concepto | Detalle |
|---|---|
| Language Model | $P(w_1, ..., w_n)$ — probabilidad de una secuencia |
| N-grama | Aproximación Markoviana: solo últimas $n-1$ palabras como contexto |
| Laplace smoothing | Añadir $k$ a todos los conteos para evitar prob=0 |
| Perplexity | $2^{-\frac{1}{N}\sum \log_2 P(w_i)}$ — menor = mejor |
| Temperatura | Controla aleatoriedad de la generación ($T<1$ = greedy, $T>1$ = creativo) |
| Detección de anomalías | Texto inusual → perplexity alta bajo el LM del dominio |

**Siguiente:** `13_information_retrieval.ipynb` — BM25, ranking y evaluación con NDCG.